<a href="https://colab.research.google.com/github/sergfer26/transformer-studio/blob/main/notebooks/vision_transformer_on_CIFAR_pytoch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vision Transformers from scratch

## 0. Vision Transformer fundamentals

### 0.1 Terminology for transfomers

| NLP Transformer | Vision Transformer (ViT) | Explanation |
|-----------------|--------------------------|-------------|
| Tokens | Patches | Words/subwords in text → image split into small fixed-size patches |
| Token IDs | Patch Indices | Each patch can be indexed like tokens |
| Token Embedding | Patch Embedding | Converts token/patch index into dense vectors via learned projection |
| Positional Embedding | Positional Embedding | Adds location information for sequence order/spatial structure |
| [CLS] Token | [CLS] Token | Special token to summarize sequence/image → used for classification |
| Encoder Input Sequence | Embedded Patch Sequence | Input to Transformer: tokens + positional encoding OR patches + positional info |


## 1. Import required libraries

In [2]:
import torch
import random
import torchvision

import numpy as np
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from torchvision import datasets, transforms


In [2]:
torch.__version__

'2.11.0+cu128'

In [3]:

torchvision.__version__

'0.26.0+cu128'

## 2. Setup Device-Agnostic Code

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [5]:
!nvidia-smi

Sun Jun 21 22:23:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
print(f"Usinf Device: {device}")

Usinf Device: cuda


## 3. Set the seed

In [5]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
random.seed(42)

## 4. Setting the hyperparameters

In [6]:
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 3e-4
PATCH_SIZE = 4
NUM_CLASSES = 10
IMAGE_SIZE = 32
CHANNELS = 3
EMBED_SIM = 256
NUM_HEADS = 8
DEPTH = 6
MLP_DIM = 512
DROPOUT_RATE = 0.1

## 5. Define Image Transformation

In [7]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
    # 1. Helps the model to converge faster
    # 2. Helps to make the numerical computations stable
])

## 6. Getting a dataset

sample_data/


In [ ]:
train_set = datasets.CIFAR10(
                              root='data',
                              train=True,
                              download=True,
                              transform=transform
                              )
# trainloader = torch.utils.data.DataLoader(dataset, batch_size=128, shuffle=True, num_workers=2)

 36%|███▌      | 61.4M/170M [34:00<1:19:15, 22.9kB/s]

In [ ]:
test_set = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)


## 7. Converting our datasets into dataloader

We do this for the following reasons:
1. **Computation efficiency**: The computer hardware may not be able to store in memory at 50_000 images in one hit. So we break it into baches of size BATCH_SIZE.
2. **Iterative training**: It gives the ANN more chances to update its gradient per epoch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')